# 🛒 E-Commerce Ad Effect Analysis — Data Generator

### What this notebook creates

| Table | Description | When |
|-------|-------------|------|
| `sellers.csv` | Seller profiles | Once |
| `items.csv` | Item listings per seller | Once |
| `ad_campaigns.csv` | Ad campaign schedules per item | Once |
| `daily_item_stats_YYYY-MM-DD.csv` | Daily performance per item | Every day (Airflow) |

### Run order
**Cell 1 → 2 → 3 → 4 → 5** in order. Click ▶ on each cell.

---
## Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

print('✅ Libraries loaded!')

---
## Cell 2 — Configuration
You can adjust the numbers here. For the first run, just leave them as-is.

In [ ]:
# ── Adjust these if needed ──────────────────────────
NUM_SELLERS    = 500       # Total number of sellers
NUM_ITEMS      = 5000      # Total number of items
AD_ITEM_RATIO  = 0.6       # 60% of items run ads at some point
OUTPUT_DIR     = '/content/ecommerce_data'
# ────────────────────────────────────────────────────

CATEGORIES = [
    'Electronics', 'Fashion', 'Beauty', 'Food',
    'Sports', 'Home & Living', 'Books', 'Baby & Toys'
]

# Price range per category (KRW)
PRICE_RANGE = {
    'Electronics':   (50000,  1500000),
    'Fashion':       (10000,  300000),
    'Beauty':        (5000,   150000),
    'Food':          (3000,   80000),
    'Sports':        (15000,  500000),
    'Home & Living': (8000,   400000),
    'Books':         (8000,   50000),
    'Baby & Toys':   (10000,  200000),
}

# Ad types you worked with
AD_TYPES = ['Recommendation', 'Category Banner']

SELLER_GRADES  = ['Basic', 'Regular', 'Power', 'Premium']
SELLER_WEIGHTS = [0.40, 0.30, 0.20, 0.10]

print('✅ Config ready!')
print(f'   Sellers: {NUM_SELLERS} | Items: {NUM_ITEMS}')
print(f'   Items running ads: ~{int(NUM_ITEMS * AD_ITEM_RATIO):,}')

---
## Cell 3 — Define Functions
This cell just registers the functions. No files are created yet.

In [ ]:
# ── Table 1: Sellers ────────────────────────────────────────────
def make_sellers():
    np.random.seed(42)
    n = NUM_SELLERS

    df = pd.DataFrame({
        'seller_id':        [f'S{str(i).zfill(4)}' for i in range(1, n+1)],
        'seller_grade':     np.random.choice(SELLER_GRADES, size=n, p=SELLER_WEIGHTS),
        'main_category':    np.random.choice(CATEGORIES, size=n),
        'joined_months_ago':np.random.randint(1, 60, size=n),   # months since joining
        'country':          np.random.choice(['KR', 'CN', 'US', 'JP'], size=n, p=[0.6, 0.2, 0.1, 0.1]),
        'avg_rating':       np.round(np.random.uniform(3.0, 5.0, size=n), 1),
        'total_items':      np.random.randint(1, 200, size=n),
    })
    return df


# ── Table 2: Items ──────────────────────────────────────────────
def make_items(sellers_df):
    np.random.seed(7)
    n = NUM_ITEMS

    # Assign sellers (weighted by their total_items)
    weights = sellers_df['total_items'].values.astype(float)
    weights /= weights.sum()
    assigned_sellers = sellers_df.sample(n=n, replace=True, weights=weights).reset_index(drop=True)

    categories = np.random.choice(CATEGORIES, size=n)
    prices = [
        round(random.randint(PRICE_RANGE[c][0]//1000, PRICE_RANGE[c][1]//1000) * 1000)
        for c in categories
    ]

    # Item registration date: between 2024-01-01 and 2024-12-31
    reg_dates = [
        (datetime(2024, 1, 1) + timedelta(days=random.randint(0, 364))).strftime('%Y-%m-%d')
        for _ in range(n)
    ]

    df = pd.DataFrame({
        'item_id':          [f'ITEM{str(i).zfill(5)}' for i in range(1, n+1)],
        'seller_id':        assigned_sellers['seller_id'].values,
        'seller_grade':     assigned_sellers['seller_grade'].values,
        'category':         categories,
        'price':            prices,
        'registered_date':  reg_dates,
        'stock':            np.random.randint(0, 500, size=n),
        'avg_rating':       np.round(np.random.uniform(2.5, 5.0, size=n), 1),
        'review_count':     np.random.randint(0, 2000, size=n),
        # AB test group — used later for ad effect comparison
        'ab_group':         np.random.choice(['A', 'B'], size=n),
    })
    return df


# ── Table 3: Ad Campaigns ───────────────────────────────────────
def make_ad_campaigns(items_df):
    """
    Each item either runs no ads, or runs 1~2 campaigns in 2025.
    This gives us clear ON/OFF periods per item for before/after analysis.
    """
    random.seed(42)
    campaigns = []

    # Pick items that will run ads
    ad_items = items_df.sample(frac=AD_ITEM_RATIO, random_state=42)

    campaign_id = 1
    for _, item in ad_items.iterrows():
        # Each item runs 1 or 2 campaigns
        num_campaigns = random.choice([1, 1, 2])  # mostly 1

        used_periods = []  # track to avoid overlapping dates
        for _ in range(num_campaigns):
            # Campaign start: random day in 2025
            start_offset = random.randint(0, 330)
            start = datetime(2025, 1, 1) + timedelta(days=start_offset)
            duration = random.randint(7, 60)  # 1 week to 2 months
            end = start + timedelta(days=duration)

            # Avoid overlap with previous campaign for same item
            overlap = any(not (end < s or start > e) for s, e in used_periods)
            if overlap:
                continue
            used_periods.append((start, end))

            ad_type = random.choice(AD_TYPES)

            # Budget: higher grade sellers spend more
            grade_multiplier = {'Basic': 1, 'Regular': 2, 'Power': 4, 'Premium': 8}
            base_budget = random.randint(50000, 200000)
            budget = base_budget * grade_multiplier.get(item['seller_grade'], 1)

            campaigns.append({
                'campaign_id':  f'CAM{str(campaign_id).zfill(6)}',
                'item_id':      item['item_id'],
                'seller_id':    item['seller_id'],
                'ad_type':      ad_type,
                'start_date':   start.strftime('%Y-%m-%d'),
                'end_date':     end.strftime('%Y-%m-%d'),
                'duration_days':duration,
                'daily_budget': budget,
            })
            campaign_id += 1

    return pd.DataFrame(campaigns)


# ── Table 4: Daily Item Stats ────────────────────────────────────
def make_daily_stats(date_str, items_df, campaigns_df):
    """
    For a given date, generate performance metrics for every item.
    Items with an active campaign on this date get a boost in impressions/clicks/orders.
    """
    target_date = datetime.strptime(date_str, '%Y-%m-%d')

    # Find which items have an active ad campaign on this date
    active_ads = campaigns_df[
        (pd.to_datetime(campaigns_df['start_date']) <= target_date) &
        (pd.to_datetime(campaigns_df['end_date'])   >= target_date)
    ][['item_id', 'ad_type', 'daily_budget']].drop_duplicates('item_id')

    # Merge items with active ad info
    df = items_df.copy()
    df = df.merge(active_ads, on='item_id', how='left')
    df['ad_active']   = df['ad_type'].notna().astype(int)   # 1 = ad running, 0 = no ad
    df['ad_type']     = df['ad_type'].fillna('None')
    df['daily_budget']= df['daily_budget'].fillna(0)

    n = len(df)

    # ── Base impressions (varies by category popularity & seller grade) ──
    grade_imp = {'Basic': 1.0, 'Regular': 1.3, 'Power': 1.8, 'Premium': 2.5}
    base_imp  = np.random.randint(100, 1000, size=n)
    grade_mult= df['seller_grade'].map(grade_imp).values

    # Ad boost: ads significantly increase impressions
    ad_imp_boost = np.where(df['ad_active'] == 1,
                            np.random.uniform(2.0, 5.0, size=n),  # 2x~5x boost
                            1.0)
    # Recommendation vs Category Banner have different impression patterns
    type_boost = np.where(df['ad_type'] == 'Recommendation',
                          np.random.uniform(1.2, 1.5, size=n),
                          np.where(df['ad_type'] == 'Category Banner',
                                   np.random.uniform(1.5, 3.0, size=n),
                                   1.0))

    impressions = (base_imp * grade_mult * ad_imp_boost * type_boost).astype(int)

    # ── Click-through rate (CTR): ad items get higher CTR ──
    base_ctr   = np.random.uniform(0.01, 0.05, size=n)   # 1%~5% organic
    ad_ctr     = np.where(df['ad_active'] == 1,
                          base_ctr * np.random.uniform(1.3, 2.0, size=n),  # ad boosts CTR
                          base_ctr)
    ad_ctr     = np.clip(ad_ctr, 0, 0.30)   # cap at 30%
    clicks     = (impressions * ad_ctr).astype(int)

    # ── Conversion rate (CVR): clicks → orders ──
    # Higher rating & review count → higher CVR
    rating_factor  = (df['avg_rating'].values - 2.5) / 2.5   # 0~1 scale
    review_factor  = np.log1p(df['review_count'].values) / np.log1p(2000)
    base_cvr       = np.random.uniform(0.02, 0.08, size=n)
    cvr            = base_cvr * (1 + 0.3 * rating_factor + 0.2 * review_factor)
    cvr            = np.clip(cvr, 0, 0.30)

    orders  = (clicks * cvr).astype(int)
    revenue = orders * df['price'].values

    # ── Assemble final table ──
    stats = pd.DataFrame({
        'date':             date_str,
        'item_id':          df['item_id'].values,
        'seller_id':        df['seller_id'].values,
        'seller_grade':     df['seller_grade'].values,
        'category':         df['category'].values,
        'price':            df['price'].values,
        'ab_group':         df['ab_group'].values,

        # Ad info
        'ad_active':        df['ad_active'].values,   # 1 or 0
        'ad_type':          df['ad_type'].values,
        'daily_budget':     df['daily_budget'].values,

        # Core metrics (the funnel)
        'impressions':      impressions,
        'clicks':           clicks,
        'orders':           orders,
        'revenue':          revenue,

        # Derived metrics
        'ctr':              np.round(np.where(impressions > 0, clicks / impressions, 0), 4),
        'conversion_rate':  np.round(np.where(clicks > 0, orders / clicks, 0), 4),
        'revenue_per_click':np.round(np.where(clicks > 0, revenue / clicks, 0), 0),
    })

    # Only keep items that had at least some impressions (remove dead stock)
    stats = stats[stats['impressions'] > 0].reset_index(drop=True)
    return stats


print('✅ All functions defined!')

---
## Cell 4 — Generate Static Tables (Run Once)
Creates `sellers.csv`, `items.csv`, and `ad_campaigns.csv`.
You only need to run this once at the start of the project.

In [ ]:
os.makedirs(f'{OUTPUT_DIR}/static',  exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/daily',   exist_ok=True)

# Sellers
print('Creating sellers...')
sellers_df = make_sellers()
sellers_df.to_csv(f'{OUTPUT_DIR}/static/sellers.csv', index=False)
print(f'  → {len(sellers_df):,} sellers saved')

# Items
print('Creating items...')
items_df = make_items(sellers_df)
items_df.to_csv(f'{OUTPUT_DIR}/static/items.csv', index=False)
print(f'  → {len(items_df):,} items saved')

# Ad Campaigns
print('Creating ad campaigns...')
campaigns_df = make_ad_campaigns(items_df)
campaigns_df.to_csv(f'{OUTPUT_DIR}/static/ad_campaigns.csv', index=False)
print(f'  → {len(campaigns_df):,} campaigns saved')

print('\n✅ Static tables done!')
print(f'   Items with ads : {campaigns_df["item_id"].nunique():,}')
print(f'   Items without  : {NUM_ITEMS - campaigns_df["item_id"].nunique():,}')

campaigns_df.head(3)

---
## Cell 5 — Generate Daily Stats

Choose **5-A** (one day, for testing) or **5-B** (full date range).

In [ ]:
# ── Cell 5-A : One day only (quick test) ───────────────────────

test_date = '2025-01-15'
df = make_daily_stats(test_date, items_df, campaigns_df)
df.to_csv(f'{OUTPUT_DIR}/daily/daily_stats_{test_date}.csv', index=False)

print(f'✅ daily_stats_{test_date}.csv saved!')
print(f'   Total items tracked : {len(df):,}')
print(f'   Items with ad active: {df["ad_active"].sum():,}')
print()
print('--- Avg metrics: Ad ON vs OFF ---')
print(df.groupby('ad_active')[['impressions','clicks','orders','revenue','conversion_rate']]
        .mean().round(2))

df.head(5)

In [ ]:
# ── Cell 5-B : Full date range ──────────────────────────────────
# Change dates as needed, then run

START_DATE = '2025-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

current    = datetime.strptime(START_DATE, '%Y-%m-%d')
end        = datetime.strptime(END_DATE,   '%Y-%m-%d')
total_days = (end - current).days + 1
all_daily  = []

print(f'Generating {total_days} days ({START_DATE} ~ {END_DATE})...')

while current <= end:
    date_str = current.strftime('%Y-%m-%d')
    df = make_daily_stats(date_str, items_df, campaigns_df)
    df.to_csv(f'{OUTPUT_DIR}/daily/daily_stats_{date_str}.csv', index=False)
    all_daily.append(df)
    current += timedelta(days=1)

# Combined file for Snowflake upload
combined = pd.concat(all_daily, ignore_index=True)
combined.to_csv(f'{OUTPUT_DIR}/daily_stats_all.csv', index=False)

print(f'\n✅ All done!')
print(f'   Total rows    : {len(combined):,}')
print(f'   Total revenue : ₩{combined["revenue"].sum():,.0f}')
print(f'   Combined file : {OUTPUT_DIR}/daily_stats_all.csv')

---
## Cell 6 — Quick Ad Effect Check
A simple sanity check to confirm the ad effect is visible in the data.

In [ ]:
# Use combined data if available, otherwise use single-day df
check_df = combined if 'combined' in dir() else df

print('=== Ad Effect Summary ===')
summary = check_df.groupby(['ad_active', 'ad_type']).agg(
    rows          = ('item_id', 'count'),
    avg_impressions = ('impressions', 'mean'),
    avg_clicks    = ('clicks', 'mean'),
    avg_orders    = ('orders', 'mean'),
    avg_ctr       = ('ctr', 'mean'),
    avg_cvr       = ('conversion_rate', 'mean'),
    avg_revenue   = ('revenue', 'mean'),
).round(2)

print(summary)

print('\n=== By Seller Grade ===')
print(check_df.groupby(['seller_grade', 'ad_active'])[['orders', 'revenue', 'conversion_rate']]
        .mean().round(2))

---
## Cell 7 — Download Files

In [ ]:
from google.colab import files

files.download(f'{OUTPUT_DIR}/static/sellers.csv')
files.download(f'{OUTPUT_DIR}/static/items.csv')
files.download(f'{OUTPUT_DIR}/static/ad_campaigns.csv')
files.download(f'{OUTPUT_DIR}/daily_stats_all.csv')

print('✅ Downloads started!')